# Octopus CUDA Tracking Demo

This notebook demonstrates executing an Octopus `TrackingTask` on a CUDA GPU and comparing the result with CPU tracking.

The notebook expects:

- Julia 1.12
- CUDA.jl installed
- a visible NVIDIA GPU
- the local Octopus source tree

In [1]:
const OCTOPUS_ROOT = dirname(@__DIR__)
const OCTOPUS_SRC = joinpath(OCTOPUS_ROOT, "src", "Octopus.jl")

include(OCTOPUS_SRC)
using .Octopus
import CUDA, Random

println("Loaded Octopus from: ", OCTOPUS_SRC)

Loaded Octopus from: /cfs/ad/dxu/Library/Julia/Octopus/src/Octopus.jl

## Check CUDA Availability

This notebook should be launched with the Julia executable that can see your GPU, for example:

```bash
/home/cfsd/dxu/.local/bin/julia --project=/home/cfsd/dxu/Library/Julia/Octopus
```

If `CUDA.functional(false)` is false, Julia and CUDA.jl are loaded, but the current notebook kernel does not have an NVIDIA device exposed. In that case the GPU cells below return a skipped status instead of treating the code as failed.


In [2]:
cuda_status = begin
    functional = CUDA.functional(false)
    visible_nodes = filter(ispath, ["/dev/nvidiactl", "/dev/nvidia0", "/dev/nvidia-uvm"])
    status = (
        julia = join(Base.julia_cmd().exec, " "),
        host = gethostname(),
        CUDA_VISIBLE_DEVICES = get(ENV, "CUDA_VISIBLE_DEVICES", "<unset>"),
        functional = functional,
        has_cuda_gpu = CUDA.has_cuda_gpu(),
        device = functional ? string(CUDA.device()) : "no CUDA device visible",
        visible_device_nodes = visible_nodes,
    )
    if !functional
        @warn "CUDA.jl is available, but this Julia process cannot initialize a CUDA device. Run this notebook from a GPU-visible session."
    else
        CUDA.versioninfo()
    end
    status
end


CUDA toolchain: 
- runtime 13.0, artifact installation
- driver 580.119.2 for 13.1
- compiler 13.1

CUDA libraries: 
- CUBLAS: 13.1.0
- CURAND: 10

.4.0
- CUFFT: 12.0.0
- CUSOLVER: 12.0.4
- CUSPARSE: 

12.6.3
- CUPTI: 2025.3.1 (API 13.0.1)
- NVML: 13.0.0+580.119.2

Julia packages: 
- CUDA: 5.9.6
- GPUArrays: 11.3.4
- GPUCompiler: 1.8.2
- KernelAbstractions: 0.9.39
- CUDA_Driver_jll: 13.1.0+2
- CUDA_Compiler_jll: 0.4.1+1
- CUDA_Runtime_jll: 0.19.2+0

Toolchain:
- Julia: 1.12.4
- LLVM: 18.1.7

1 device:
  0: NVIDIA RTX 4500 Ada Generation (sm_89, 23.300 GiB / 23.994 GiB available)


(julia = "/cfs/ad/dxu/packages/julias/julia-1.12.4/bin/julia -C native -J/cfs/ad/dxu/packages/julias/julia-1.12.4/lib/julia/sys.so -g1 --color=yes", host = "acnlinj4.pbn.bnl.gov", CUDA_VISIBLE_DEVICES = "<unset>", functional = true, has_cuda_gpu = true, device = "CUDA.CuDevice(0)", visible_device_nodes = ["/dev/nvidiactl", "/dev/nvidia0", "/dev/nvidia-uvm"])

## Define a Multi-Element Task

The task is defined at the specification layer first. It is compiled to runtime tracking maps only when `execute!` is called.

In [3]:
line = (
    CrabDispersionSpec{Float32}(zeta1 = 0.1f0),
    MomentumDispersionSpec{Float32}(eta1 = 0.2f0),
)

cpu_task = TrackingTask(line; policy = CPUThreadsExecutionPolicy())
gpu_task = TrackingTask(line; policy = GPUExecutionPolicy())

(
    elements = name.(gpu_task.elements),
    policy = typeof(gpu_task.policy),
    contracts = gpu_task.contracts,
)

(elements = (:crab_dispersion, :momentum_dispersion), policy = GPUExecutionPolicy, contracts = DataType[ElementTrackingBackendConsistencyContract])

## Run the CPU Reference

The CPU and GPU examples use the same initial coordinates and the same task structure.

In [4]:
cpu_rep = Phase6DRep(
    Float32[1.0, 0.2],
    Float32[2.0, 0.1],
    Float32[3.0, 0.3],
    Float32[4.0, 0.4],
    Float32[5.0, 0.5],
    Float32[6.0, 0.6],
)

execute!(cpu_task, cpu_rep; turns = 2)

cpu_result = [cpu_rep[i] for i in keys(cpu_rep)]

2-element Vector{NTuple{6, Float32}}:
 (4.32, 2.0, 3.0, 4.0, 5.8, 5.6000004)
 (0.536, 0.1, 0.3, 0.4, 0.53999996, 0.58000004)

## Run on CUDA

`Phase6DRep` stores CUDA arrays on the host side. Octopus converts them to kernel-safe device views before launching the CUDA tracking kernel.

In [5]:
if CUDA.functional(false)
    gpu_rep = Phase6DRep(
        CUDA.CuArray(Float32[1.0, 0.2]),
        CUDA.CuArray(Float32[2.0, 0.1]),
        CUDA.CuArray(Float32[3.0, 0.3]),
        CUDA.CuArray(Float32[4.0, 0.4]),
        CUDA.CuArray(Float32[5.0, 0.5]),
        CUDA.CuArray(Float32[6.0, 0.6]),
    )

    execute!(gpu_task, gpu_rep; turns = 2)
    CUDA.synchronize()

    gpu_result = collect(zip(
        Array(gpu_rep.x),
        Array(gpu_rep.px),
        Array(gpu_rep.y),
        Array(gpu_rep.py),
        Array(gpu_rep.z),
        Array(gpu_rep.pz),
    ))
else
    gpu_rep = nothing
    gpu_result = nothing
    @warn "Skipping CUDA run because no CUDA device is visible to this notebook kernel."
end


2-element Vector{NTuple{6, Float32}}:
 (4.32, 2.0, 3.0, 4.0, 5.8, 5.6000004)
 (0.536, 0.1, 0.3, 0.4, 0.53999996, 0.58000004)

## Compare CPU and GPU Results

In [6]:
if gpu_rep === nothing
    (status = :skipped, reason = "no CUDA device visible")
else
    cpu_arrays = (cpu_rep.x, cpu_rep.px, cpu_rep.y, cpu_rep.py, cpu_rep.z, cpu_rep.pz)
    gpu_arrays = (Array(gpu_rep.x), Array(gpu_rep.px), Array(gpu_rep.y), Array(gpu_rep.py), Array(gpu_rep.z), Array(gpu_rep.pz))

    max_errors = map((cpu, gpu) -> maximum(abs.(cpu .- gpu)), cpu_arrays, gpu_arrays)

    (
        cpu_result = cpu_result,
        gpu_result = gpu_result,
        max_coordinate_errors = max_errors,
        max_error = maximum(max_errors),
    )
end


(cpu_result = NTuple{6, Float32}[(4.32, 2.0, 3.0, 4.0, 5.8, 5.6000004), (0.536, 0.1, 0.3, 0.4, 0.53999996, 0.58000004)], gpu_result = NTuple{6, Float32}[(4.32, 2.0, 3.0, 4.0, 5.8, 5.6000004), (0.536, 0.1, 0.3, 0.4, 0.53999996, 0.58000004)], max_coordinate_errors = (0.0f0, 0.0f0, 0.0f0, 0.0f0, 0.0f0, 0.0f0), max_error = 0.0f0)

## Nonlinear Thin Crab Cavity Task

`ThinCrabCavitySpec` is a zero-length symplectic kick, so it uses `Symplectic6DMap`.

In [7]:
cavity_line = (
    ThinCrabCavitySpec{2}(1.0f8; strengthX = (1.0f0, 0.5f0), strengthY = (0.25f0, 0.125f0), phase = (0.0f0, 0.1f0)),
)

cavity_cpu_task = TrackingTask(cavity_line; policy = CPUThreadsExecutionPolicy())
cavity_gpu_task = TrackingTask(cavity_line; policy = GPUExecutionPolicy())

cavity_cpu = Phase6DRep(Float32[1.0, 0.2], Float32[2.0, 0.1], Float32[3.0, 0.3], Float32[4.0, 0.4], Float32[0.01, 0.02], Float32[6.0, 0.6])
execute!(cavity_cpu_task, cavity_cpu; turns = 1)

if CUDA.functional(false)
    cavity_gpu = Phase6DRep(CUDA.CuArray(Float32[1.0, 0.2]), CUDA.CuArray(Float32[2.0, 0.1]), CUDA.CuArray(Float32[3.0, 0.3]), CUDA.CuArray(Float32[4.0, 0.4]), CUDA.CuArray(Float32[0.01, 0.02]), CUDA.CuArray(Float32[6.0, 0.6]))
    execute!(cavity_gpu_task, cavity_gpu; turns = 1)
    CUDA.synchronize()
    cavity_errors = map((cpu, gpu) -> maximum(abs.(cpu .- gpu)),
        (cavity_cpu.x, cavity_cpu.px, cavity_cpu.y, cavity_cpu.py, cavity_cpu.z, cavity_cpu.pz),
        (Array(cavity_gpu.x), Array(cavity_gpu.px), Array(cavity_gpu.y), Array(cavity_gpu.py), Array(cavity_gpu.z), Array(cavity_gpu.pz)))
    (status = :ran, max_coordinate_errors = cavity_errors, max_error = maximum(cavity_errors))
else
    (status = :skipped, reason = "no CUDA device visible", cpu_reference_checksum = sum(cavity_cpu.x) + sum(cavity_cpu.px) + sum(cavity_cpu.y) + sum(cavity_cpu.py) + sum(cavity_cpu.z) + sum(cavity_cpu.pz))
end


(status = :ran, max_coordinate_errors = (0.0f0, 0.0f0, 0.0f0, 0.0f0, 0.0f0, 0.0f0), max_error = 0.0f0)

## One-Million-Particle Wall-Time Benchmark

This benchmark tracks 1,000,000 `Float32` particles through a three-element line for 10 turns. CPU and GPU runs start from the same initial coordinate arrays. The timing excludes first kernel compilation by running a small warm-up before measuring.

CPU thread count is fixed when Julia starts. To compare one thread and multiple threads, start Julia with different thread counts, for example `julia --threads=1` and `julia --threads=8`. Use the same random seed in each process so the initial coordinates match.

The helper cells below report both wall time and computation speed in particle-turns per second. They also return final coordinate arrays so CPU and GPU results can be compared from identical starts.

Same-start command-line measurements on this machine were:

| backend | threads/device | particles | turns | wall time (s) | particle-turns/s | checksum | max CPU/GPU error |
|---|---:|---:|---:|---:|---:|---:|---:|
| CPU | 1 thread | 1,000,000 | 10 | 0.222182 | 4.50e7 | 1.8732421e6 | - |
| CPU | 8 threads | 1,000,000 | 10 | 0.028875 | 3.46e8 | 1.8732421e6 | - |
| CUDA | GPU 0 | 1,000,000 | 10 | 0.050790 | 1.97e8 | 1.8732422e6 | 1.43e-6 |

In [8]:
function benchmark_line_specs()
    return (
        CrabDispersionSpec{Float32}(zeta1 = 0.1f0),
        MomentumDispersionSpec{Float32}(eta1 = 0.2f0),
        XYCouplingSpec{Float32}(r1 = 0.01f0),
    )
end

function make_initial_coordinates(N; seed = 1234)
    Random.seed!(seed)
    return ntuple(_ -> rand(Float32, N), 6)
end

function cpu_rep_from(initial)
    return Phase6DRep(copy(initial[1]), copy(initial[2]), copy(initial[3]),
                      copy(initial[4]), copy(initial[5]), copy(initial[6]))
end

function gpu_rep_from(initial)
    return Phase6DRep(CUDA.CuArray(initial[1]), CUDA.CuArray(initial[2]), CUDA.CuArray(initial[3]),
                      CUDA.CuArray(initial[4]), CUDA.CuArray(initial[5]), CUDA.CuArray(initial[6]))
end

function host_arrays(rep::Phase6DRep)
    return (Array(rep.x), Array(rep.px), Array(rep.y), Array(rep.py), Array(rep.z), Array(rep.pz))
end

function benchmark_cpu_current_threads(initial = make_initial_coordinates(1_000_000); turns = 10)
    N = length(initial[1])
    bench_line = benchmark_line_specs()
    task = TrackingTask(bench_line; policy = CPUThreadsExecutionPolicy())

    warm = Phase6DRep(
        rand(Float32, 1024), rand(Float32, 1024), rand(Float32, 1024),
        rand(Float32, 1024), rand(Float32, 1024), rand(Float32, 1024),
    )
    execute!(task, warm; turns = 1)

    rep = cpu_rep_from(initial)

    t0 = time()
    execute!(task, rep; turns = turns)
    seconds = time() - t0

    return (
        backend = "CPU",
        threads = Threads.nthreads(),
        particles = N,
        turns = turns,
        seconds = seconds,
        particle_turns_per_second = N * turns / seconds,
        checksum = sum(rep.x) + sum(rep.pz),
        final = host_arrays(rep),
    )
end

benchmark_cpu_current_threads (generic function with 2 methods)

In [9]:
initial = make_initial_coordinates(1_000_000; seed = 1234)
cpu_benchmark = benchmark_cpu_current_threads(initial; turns = 10)

NamedTuple{(:backend, :threads, :particles, :turns, :seconds, :particle_turns_per_second, :checksum)}(cpu_benchmark)

(backend = "CPU", threads = 1, particles = 1000000, turns = 10, seconds = 0.14052391052246094, particle_turns_per_second = 7.116226671191041e7, checksum = 1.8732421f6)

In [10]:
function benchmark_gpu(initial = make_initial_coordinates(1_000_000); turns = 10)
    CUDA.functional() || error("No functional CUDA device is visible to this Julia session.")

    N = length(initial[1])
    bench_line = benchmark_line_specs()
    task = TrackingTask(bench_line; policy = GPUExecutionPolicy())

    warm = Phase6DRep(
        CUDA.rand(Float32, 1024), CUDA.rand(Float32, 1024), CUDA.rand(Float32, 1024),
        CUDA.rand(Float32, 1024), CUDA.rand(Float32, 1024), CUDA.rand(Float32, 1024),
    )
    execute!(task, warm; turns = 1)
    CUDA.synchronize()

    rep = gpu_rep_from(initial)
    CUDA.synchronize()

    t0 = time()
    execute!(task, rep; turns = turns)
    CUDA.synchronize()
    seconds = time() - t0

    return (
        backend = "CUDA",
        device = string(CUDA.device()),
        particles = N,
        turns = turns,
        seconds = seconds,
        particle_turns_per_second = N * turns / seconds,
        checksum = sum(Array(rep.x)) + sum(Array(rep.pz)),
        final = host_arrays(rep),
    )
end

benchmark_gpu (generic function with 2 methods)

In [11]:
if CUDA.functional(false)
    gpu_benchmark = benchmark_gpu(initial; turns = 10)
    NamedTuple{(:backend, :device, :particles, :turns, :seconds, :particle_turns_per_second, :checksum)}(gpu_benchmark)
else
    gpu_benchmark = nothing
    (status = :skipped, reason = "no CUDA device visible")
end


(backend = "CUDA", device = "CUDA.CuDevice(0)", particles = 1000000, turns = 10, seconds = 0.030740976333618164, particle_turns_per_second = 3.2529871177396715e8, checksum = 1.8732422f6)

In [12]:
if gpu_benchmark === nothing
    (status = :skipped, reason = "no CUDA device visible", same_initial_coordinates = true)
else
    same_start_errors = map((cpu, gpu) -> maximum(abs.(cpu .- gpu)),
                            cpu_benchmark.final, gpu_benchmark.final)

    (
        same_initial_coordinates = true,
        cpu_seconds = cpu_benchmark.seconds,
        gpu_seconds = gpu_benchmark.seconds,
        cpu_particle_turns_per_second = cpu_benchmark.particle_turns_per_second,
        gpu_particle_turns_per_second = gpu_benchmark.particle_turns_per_second,
        gpu_speedup_vs_cpu_current_threads = cpu_benchmark.seconds / gpu_benchmark.seconds,
        throughput_ratio_gpu_vs_cpu_current_threads = gpu_benchmark.particle_turns_per_second / cpu_benchmark.particle_turns_per_second,
        max_coordinate_errors = same_start_errors,
        max_error = maximum(same_start_errors),
    )
end


(same_initial_coordinates = true, cpu_seconds = 0.14052391052246094, gpu_seconds = 0.030740976333618164, cpu_particle_turns_per_second = 7.116226671191041e7, gpu_particle_turns_per_second = 3.2529871177396715e8, gpu_speedup_vs_cpu_current_threads = 4.5712247066396765, throughput_ratio_gpu_vs_cpu_current_threads = 4.571224706639677, max_coordinate_errors = (1.1920929f-6, 1.1920929f-6, 2.9802322f-7, 0.0f0, 1.4305115f-6, 5.9604645f-7), max_error = 1.4305115f-6)

## Strong-Beam CUDA Consistency and Speed

These cells compare CPU and CUDA tracking for weak-strong beam-beam elements. CPU and GPU start from identical coordinate arrays. The report includes absolute error, relative error, wall time, and particle-turn throughput.

The GPU path uses the device-compatible Faddeeva approximation in the CUDA kernel; the CPU reference uses the normal Octopus CPU tracking path. If the notebook kernel has no visible CUDA device, the CUDA part returns `status = :skipped`.


In [14]:
function strong_beam_initial(N; T = Float64, seed = 20240711)
    Random.seed!(seed)
    scale = T(1e-6)
    return (
        scale .* randn(T, N),
        scale .* randn(T, N),
        scale .* randn(T, N),
        scale .* randn(T, N),
        scale .* randn(T, N),
        scale .* randn(T, N),
    )
end

strong_beam_cpu_rep(initial) = Phase6DRep(
    copy(initial[1]), copy(initial[2]), copy(initial[3]),
    copy(initial[4]), copy(initial[5]), copy(initial[6]),
)

strong_beam_gpu_rep(initial) = Phase6DRep(
    CUDA.CuArray(initial[1]), CUDA.CuArray(initial[2]), CUDA.CuArray(initial[3]),
    CUDA.CuArray(initial[4]), CUDA.CuArray(initial[5]), CUDA.CuArray(initial[6]),
)

coordinate_labels() = (:x, :px, :y, :py, :z, :pz)
coordinate_tuple(rep) = (copy(rep.x), copy(rep.px), copy(rep.y), copy(rep.py), copy(rep.z), copy(rep.pz))
gpu_coordinate_tuple(rep) = (Array(rep.x), Array(rep.px), Array(rep.y), Array(rep.py), Array(rep.z), Array(rep.pz))

function coordinate_error_report(cpu_final, gpu_final)
    abs_errors = map((cpu, gpu) -> maximum(abs.(cpu .- gpu)), cpu_final, gpu_final)
    scales = map(cpu -> max(maximum(abs.(cpu)), eps(eltype(cpu))), cpu_final)
    rel_errors = map(/, abs_errors, scales)
    return (
        labels = coordinate_labels(),
        max_absolute_errors = abs_errors,
        max_relative_errors = rel_errors,
        max_absolute_error = maximum(abs_errors),
        max_relative_error = maximum(rel_errors),
    )
end

function timed_execute!(task, rep; turns, synchronize_cuda = false)
    synchronize_cuda && CUDA.synchronize()
    t0 = time()
    execute!(task, rep; turns = turns)
    synchronize_cuda && CUDA.synchronize()
    return time() - t0
end

function thin_strong_beam_spec(; T = Float64)
    return ThinStrongBeamSpec{T}(;
        kbb = T(1.0e-3),
        klum = one(T),
        beta = (T(0.1), T(0.12)),
        sigma = (T(2.0e-6), T(3.0e-6)),
        center = (zero(T), zero(T), zero(T)),
        angle = (zero(T), zero(T), zero(T)),
        dynamic_drift_flag = 0,
    )
end

function strong_beam_consistency(; N = 100_000, turns = 1, T = Float64, seed = 20240711)
    spec = thin_strong_beam_spec(; T = T)
    initial = strong_beam_initial(N; T = T, seed = seed)

    cpu_rep = strong_beam_cpu_rep(initial)
    cpu_task = TrackingTask((spec,); policy = CPUThreadsExecutionPolicy())
    cpu_seconds = timed_execute!(cpu_task, cpu_rep; turns = turns)
    cpu_final = coordinate_tuple(cpu_rep)

    base = (
        element = :thin_strong_beam,
        particles = N,
        turns = turns,
        cpu_seconds = cpu_seconds,
        cpu_particle_turns_per_second = N * turns / cpu_seconds,
        cpu_checksum = sum(sum, cpu_final),
    )

    if !CUDA.functional(false)
        return merge(base, (status = :skipped, reason = "no CUDA device visible"))
    end

    gpu_rep = strong_beam_gpu_rep(initial)
    gpu_task = TrackingTask((spec,); policy = GPUExecutionPolicy())
    gpu_seconds = timed_execute!(gpu_task, gpu_rep; turns = turns, synchronize_cuda = true)
    gpu_final = gpu_coordinate_tuple(gpu_rep)
    errors = coordinate_error_report(cpu_final, gpu_final)

    return merge(base, errors, (
        status = :ran,
        device = string(CUDA.device()),
        gpu_seconds = gpu_seconds,
        gpu_particle_turns_per_second = N * turns / gpu_seconds,
        gpu_speedup_vs_cpu = cpu_seconds / gpu_seconds,
        gpu_checksum = sum(sum, gpu_final),
    ))
end

strong_beam_consistency()


(element = :thin_strong_beam, particles = 100000, turns = 1, cpu_seconds = 0.07337093353271484, cpu_particle_turns_per_second = 1.3629375446805744e6, cpu_checksum = 1.0412856544957758e9, labels = (:x, :px, :y, :py, :z, :pz), max_absolute_errors = (5.987337090961747e-17, 6.204518669672421e-11, 8.001412032943023e-17, 6.073719305277336e-11, 0.0, 8.185452315956354e-9), max_relative_errors = (8.869816325873923e-14, 1.646421502515165e-13, 1.5982574558080748e-13, 1.7500794263066557e-13, 0.0, 2.304642615838823e-13), max_absolute_error = 8.185452315956354e-9, max_relative_error = 2.304642615838823e-13, status = :ran, device = "CUDA.CuDevice(0)", gpu_seconds = 0.0328521728515625, gpu_particle_turns_per_second = 3.043938690199721e6, gpu_speedup_vs_cpu = 2.233366233163028, gpu_checksum = 1.0412856544957726e9)

## Gaussian Strong-Beam Multi-Slice Check

This cell tests `GaussianStrongBeamSpec` with multiple longitudinal slices. It uses the same reporting style as the thin strong-beam check: same initial coordinates, CPU/GPU speed, absolute error, and relative error.


In [17]:
function gaussian_strong_beam_spec(; T = Float64, ns = 7)
    thin = thin_strong_beam_spec(; T = T)
    return GaussianStrongBeamSpec{T}(;
        thin = thin,
        ns = ns,
        sigz = T(4.0e-3),
        slice_method = :equal_area,
    )
end

function gaussian_strong_beam_consistency(; N = 1_000_000, turns = 1, T = Float64, ns = 7, seed = 20240712)
    spec = gaussian_strong_beam_spec(; T = T, ns = ns)
    initial = strong_beam_initial(N; T = T, seed = seed)

    cpu_rep = strong_beam_cpu_rep(initial)
    cpu_task = TrackingTask((spec,); policy = CPUThreadsExecutionPolicy())
    cpu_seconds = timed_execute!(cpu_task, cpu_rep; turns = turns)
    cpu_final = coordinate_tuple(cpu_rep)

    base = (
        element = :gaussian_strong_beam,
        slices = ns,
        particles = N,
        turns = turns,
        cpu_seconds = cpu_seconds,
        cpu_particle_turns_per_second = N * turns / cpu_seconds,
        cpu_checksum = sum(sum, cpu_final),
    )

    if !CUDA.functional(false)
        return merge(base, (status = :skipped, reason = "no CUDA device visible"))
    end

    gpu_rep = strong_beam_gpu_rep(initial)
    gpu_task = TrackingTask((spec,); policy = GPUExecutionPolicy())
    gpu_seconds = timed_execute!(gpu_task, gpu_rep; turns = turns, synchronize_cuda = true)
    gpu_final = gpu_coordinate_tuple(gpu_rep)
    errors = coordinate_error_report(cpu_final, gpu_final)

    return merge(base, errors, (
        status = :ran,
        device = string(CUDA.device()),
        gpu_seconds = gpu_seconds,
        gpu_particle_turns_per_second = N * turns / gpu_seconds,
        gpu_speedup_vs_cpu = cpu_seconds / gpu_seconds,
        gpu_checksum = sum(sum, gpu_final),
    ))
end

gaussian_strong_beam_consistency()


(element = :gaussian_strong_beam, slices = 7, particles = 1000000, turns = 1, cpu_seconds = 2.0840909481048584, cpu_particle_turns_per_second = 479825.50901117694, cpu_checksum = 2.1264232634497237e8, labels = (:x, :px, :y, :py, :z, :pz), max_absolute_errors = (1.497621471280297e-13, 9.514433685353652e-11, 1.5851209234085673e-13, 1.0015099860538612e-10, 0.0, 3.5992542279927875e-10), max_relative_errors = (9.491218510472431e-13, 1.7671614189562801e-12, 1.0904848037700192e-12, 2.019305108442903e-12, 0.0, 4.966572770033658e-13), max_absolute_error = 3.5992542279927875e-10, max_relative_error = 2.019305108442903e-12, status = :ran, device = "CUDA.CuDevice(0)", gpu_seconds = 0.021059036254882812, gpu_particle_turns_per_second = 4.748555384476044e7, gpu_speedup_vs_cpu = 98.96421293361108, gpu_checksum = 2.1264232634497178e8)

## Strong-Beam GPU Timing Breakdown

This cell warms the CUDA kernels first, then separates host/device setup, kernel execution, current host-side luminosity reduction, final coordinate copy, and the high-level `execute!` timing. It uses internal kernel entry points so the breakdown is diagnostic rather than the public API.


In [18]:
function gpu_timing_breakdown(; element = :thin, N = 500_000, turns = 1, T = Float64, ns = 7, seed = 20240713,
                              threads = 256, blocks = 256)
    if !CUDA.functional(false)
        return (status = :skipped, reason = "no CUDA device visible")
    end

    spec = element == :thin ? thin_strong_beam_spec(; T = T) : gaussian_strong_beam_spec(; T = T, ns = ns)
    runtime = compile_runtime(spec)
    initial = strong_beam_initial(N; T = T, seed = seed)

    # Warm up compilation and CUDA runtime using a small representative problem.
    warm_initial = strong_beam_initial(4096; T = T, seed = seed + 1)
    warm_rep = strong_beam_gpu_rep(warm_initial)
    warm_task = TrackingTask((spec,); policy = GPUExecutionPolicy(threads = threads, blocks = blocks))
    execute!(warm_task, warm_rep; turns = 1)
    CUDA.synchronize()

    setup_t0 = time()
    rep = strong_beam_gpu_rep(initial)
    lum = CUDA.zeros(T, N)
    if element == :gaussian
        slice_center = CUDA.CuArray(T.(runtime.slice_center))
        slice_weight = CUDA.CuArray(T.(runtime.slice_weight))
        slice_hoffset = CUDA.CuArray(T.(runtime.slice_hoffset))
        slice_voffset = CUDA.CuArray(T.(runtime.slice_voffset))
    end
    CUDA.synchronize()
    setup_seconds = time() - setup_t0

    kernel_t0 = time()
    if element == :thin
        e = runtime
        CUDA.@cuda threads=threads blocks=blocks Octopus.cuda_track_thin_strong_beam_kernel!(
            rep, lum, Int(turns),
            e.sigx0, e.sigy0, e.betx0, e.bety0, e.alfx0, e.alfy0,
            e.gamx0, e.gamy0, e.emitx, e.emity, e.kbb, e.klum,
            e.xo, e.yo, e.zo, e.pxo, e.pyo, e.pzo,
            e.ppxo, e.ppyo, e.ppzo, e.dynamic_drift_flag,
        )
    else
        e = runtime
        t = e.thin
        CUDA.@cuda threads=threads blocks=blocks Octopus.cuda_track_gaussian_strong_beam_kernel!(
            rep, lum, Int(turns), e.ns, slice_center, slice_weight, slice_hoffset, slice_voffset,
            t.sigx0, t.sigy0, t.betx0, t.bety0, t.alfx0, t.alfy0,
            t.gamx0, t.gamy0, t.emitx, t.emity, t.kbb, t.klum,
            t.xo, t.yo, t.zo, t.pxo, t.pyo, t.pzo,
            t.ppxo, t.ppyo, t.ppzo, t.dynamic_drift_flag,
        )
    end
    CUDA.synchronize()
    kernel_seconds = time() - kernel_t0

    reduction_t0 = time()
    luminosity = sum(Array(lum))
    reduction_seconds = time() - reduction_t0

    copy_t0 = time()
    final = gpu_coordinate_tuple(rep)
    copy_seconds = time() - copy_t0

    # Compare with the public high-level path after warm-up.
    public_rep = strong_beam_gpu_rep(initial)
    public_task = TrackingTask((spec,); policy = GPUExecutionPolicy(threads = threads, blocks = blocks))
    public_t0 = time()
    execute!(public_task, public_rep; turns = turns)
    CUDA.synchronize()
    public_execute_seconds = time() - public_t0

    return (
        status = :ran,
        element = element,
        particles = N,
        turns = turns,
        slices = element == :gaussian ? ns : 1,
        device = string(CUDA.device()),
        setup_seconds = setup_seconds,
        kernel_seconds = kernel_seconds,
        kernel_particle_turns_per_second = N * turns / kernel_seconds,
        luminosity_reduction_seconds = reduction_seconds,
        final_coordinate_copy_seconds = copy_seconds,
        public_execute_seconds = public_execute_seconds,
        public_particle_turns_per_second = N * turns / public_execute_seconds,
        luminosity = luminosity,
        checksum = sum(sum, final),
    )
end

(
    thin = gpu_timing_breakdown(element = :thin, N = 500_000, turns = 1),
    gaussian = gpu_timing_breakdown(element = :gaussian, N = 500_000, turns = 1, ns = 7),
)


(thin = (status = :ran, element = :thin, particles = 500000, turns = 1, slices = 1, device = "CUDA.CuDevice(0)", setup_seconds = 0.0202639102935791, kernel_seconds = 0.0556790828704834, kernel_particle_turns_per_second = 8.9800329715032e6, luminosity_reduction_seconds = 0.014379024505615234, final_coordinate_copy_seconds = 0.004428863525390625, public_execute_seconds = 0.005721092224121094, public_particle_turns_per_second = 8.739589931655276e7, luminosity = 1.1257164463943554e16, checksum = 5.206194721100592e9), gaussian = (status = :ran, element = :gaussian, particles = 500000, turns = 1, slices = 7, device = "CUDA.CuDevice(0)", setup_seconds = 0.025443077087402344, kernel_seconds = 0.06545805931091309, kernel_particle_turns_per_second = 7.638478825427698e6, luminosity_reduction_seconds = 0.0014739036560058594, final_coordinate_copy_seconds = 0.004449129104614258, public_execute_seconds = 0.009211063385009766, public_particle_turns_per_second = 5.4282549050059535e7, luminosity = 1.60

In [ ]:
ElementSpec{:thin_crab_cavity}

In [ ]:
registry = build_registry()

In [ ]:
parameter_schema(registry.elements[1])

In [ ]:
example_spec(registry.elements[1])

In [ ]:
construction_help(registry.elements[1])

In [ ]:
?CrabDispersionSpec

In [ ]:
lb=example_spec(LorentzBoostSpec)
params(lb)

In [ ]:
?kind

In [ ]:
element_help()

In [ ]:
element_help(:crab_dispersion)

In [ ]:
element_help(:momentum_dispersion)